# Hungarian Algorithm：从 KKT 条件到精确匹配实现

本笔记讨论多目标跟踪（MOT）中的 Track–Detection 数据关联。目标是：给定代价矩阵，求一对一匹配，使总代价最小。

学习目标：

1. 将 assignment 写成整数线性规划及其 LP 松弛；
2. 推导对偶问题与 KKT 最优条件；
3. 理解 Hungarian algorithm 如何维护对偶变量与零松弛边；
4. 手算一个例子，并实现不依赖 SciPy 的精确算法。

## 1. Assignment 的线性规划形式

设有 $n$ 条 Track 和 $n$ 个 Detection，$c_{ij}$ 表示将 Track $i$ 匹配到 Detection $j$ 的代价。令 $x_{ij}=1$ 表示选择该边，反之为 $0$。

$$\begin{aligned}
\min_{X}\quad & \sum_{i=1}^n\sum_{j=1}^n c_{ij}x_{ij}\\
\text{s.t.}\quad & \sum_{j=1}^n x_{ij}=1,\quad i=1,\ldots,n\\
& \sum_{i=1}^n x_{ij}=1,\quad j=1,\ldots,n\\
& x_{ij}\in\{0,1\}.
\end{aligned}$$

前两组约束分别表示每条 Track 和每个 Detection 都恰好参与一次匹配。分析时先放松整数条件为 $x_{ij}\geq0$。该 LP 的约束矩阵是全酉模（totally unimodular）的；等价地，可行域是双随机矩阵的集合，其极点是置换矩阵（Birkhoff–von Neumann theorem）。所以 LP 最优极点天然是 $0/1$ 解，Hungarian 是**精确算法而非近似算法**。

## 2. 对偶与 KKT 条件

给每个行约束分配对偶变量 $u_i$，每个列约束分配对偶变量 $v_j$。LP 对偶为：

$$\begin{aligned}
\max_{u,v}\quad &\sum_i u_i+\sum_jv_j\\
\text{s.t.}\quad &u_i+v_j\leq c_{ij},\quad \forall i,j.
\end{aligned}$$

定义 reduced cost / slack：

$$r_{ij}=c_{ij}-u_i-v_j.$$

KKT 条件在这里既必要又充分：

- 原始可行性：$x_{ij}\geq0$，且每行、每列和都为 $1$；
- 对偶可行性：$r_{ij}\geq0$；
- 互补松弛：

$$\boxed{x_{ij}r_{ij}=x_{ij}(c_{ij}-u_i-v_j)=0.}$$

因此，被选中的匹配边必须满足 $c_{ij}=u_i+v_j$，也就是 $r_{ij}=0$。这些边称为 tight edges，组成 equality graph（零松弛图）。若在零松弛图中找到完美匹配，则

$$\sum_{ij}c_{ij}x_{ij}=\sum_i u_i+\sum_jv_j,$$

原始目标与对偶目标相等，由强对偶性，该匹配全局最优。

## 3. Hungarian 的原始—对偶流程

算法维护对偶可行的 $u,v$ 以及零松弛边上的当前匹配。

1. 初始化：$u_i=\min_jc_{ij},\ v_j=0$。于是 $u_i+v_j\leq c_{ij}$，且每行至少有一条零松弛边。
2. 在 equality graph 中寻找增广路径；找到后翻转路径上的匹配/非匹配边，使匹配数增加 1。
3. 若从某个未匹配行出发的交替树无法继续，记树内行、列集合为 $S,T$。计算

$$\delta=\min_{i\in S,\ j\notin T}(c_{ij}-u_i-v_j).$$

并更新

$$u_i\leftarrow u_i+\delta\ (i\in S),\qquad v_j\leftarrow v_j-\delta\ (j\in T).$$

这不会破坏 $r_{ij}\geq0$，且至少使一条 $S\to \bar T$ 边变成零松弛边。继续扩展交替树。对从未匹配行开始的树，$|S|=|T|+1$，故对偶目标增加 $\delta(|S|-|T|)=\delta>0$。
4. 重复直到匹配大小为 $n$。此时完美匹配仅使用 tight edges，满足 KKT，因而最优。

In [ ]:
import numpy as np

# 行 = Track，列 = Detection；数值可理解为 1 - IoU 或融合后的 association cost
C = np.array([
    [4., 1., 3.],
    [2., 0., 5.],
    [3., 2., 2.],
])
C

## 4. 数值例子：先观察 KKT 量

对上面的代价矩阵，按行最小值初始化：

$$u=[1,0,2]^T,\qquad v=[0,0,0]^T.$$

此时 slack matrix 是 $R=C-u\mathbf{1}^T-\mathbf{1}v^T$。值为零的位置就是初始 equality graph 的边。该图尚不能让每个 Track 都匹配不同 Detection，因此必须利用交替树和 $\delta$ 更新增加零边。下面的实现会记录每一次对偶更新；最终解应为 $(0\to1,1\to0,2\to2)$，总代价为 $1+2+2=5$。

In [ ]:
u0 = C.min(axis=1)
v0 = np.zeros(C.shape[1])
R0 = C - u0[:, None] - v0[None, :]
print('u =', u0)
print('v =', v0)
print('initial slack R =\n', R0)
print('tight edges (row, col) =', list(zip(*np.where(np.isclose(R0, 0)))))

## 5. 纯 NumPy 的 Hungarian 实现

下面实现采用经典的短增广路径（shortest augmenting path）写法。数组 `u`、`v` 是对偶势函数；`minv[j]` 保存交替树到尚未访问列 $j$ 的最小 slack；每轮选取最小的 $\delta$，这正是上一节的对偶更新。

代码要求行数不大于列数。MOT 中若 Track 与 Detection 数量不同，通常先通过 dummy 节点补成方阵，或只对可匹配部分构造矩阵；后面会给出一个便利包装器。

In [ ]:
def hungarian_min_cost(cost, verbose=False):
    """精确求解 row <= col 的最小代价一对一匹配。返回 row_ind, col_ind, u, v。"""
    cost = np.asarray(cost, dtype=float)
    n_rows, n_cols = cost.shape
    if n_rows > n_cols:
        raise ValueError('此实现要求行数不大于列数；请转置或先补 dummy 列。')
    if not np.isfinite(cost).all():
        raise ValueError('请用较大的有限代价表示禁止边，而不是 inf。')

    # 使用 1-based 下标，使 p[j] 表示与列 j 匹配的行；p[0] 是当前增广路径的起点。
    u = np.zeros(n_rows + 1)
    v = np.zeros(n_cols + 1)
    p = np.zeros(n_cols + 1, dtype=int)
    way = np.zeros(n_cols + 1, dtype=int)

    for i in range(1, n_rows + 1):
        p[0] = i
        j0 = 0
        minv = np.full(n_cols + 1, np.inf)
        used = np.zeros(n_cols + 1, dtype=bool)

        # 建立从未匹配行 i 出发的交替树，直到抵达未匹配列。
        while True:
            used[j0] = True
            i0 = p[j0]
            delta = np.inf
            j1 = 0
            for j in range(1, n_cols + 1):
                if not used[j]:
                    cur = cost[i0 - 1, j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur
                        way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]
                        j1 = j

            # 对偶更新：树内行 u 加 delta，树内列 v 减 delta。
            for j in range(n_cols + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            if verbose:
                print(f'augment row {i-1}: delta={delta:.3f}, new tight column={j1-1}')

            j0 = j1
            if p[j0] == 0:  # 到达未匹配列，得到一条增广路径
                break

        # 沿 way 回溯，并翻转增广路径上的边。
        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break

    row_ind = p[1:] - 1
    col_ind = np.arange(n_cols)
    keep = row_ind >= 0
    row_ind, col_ind = row_ind[keep], col_ind[keep]
    order = np.argsort(row_ind)
    return row_ind[order], col_ind[order], u[1:], v[1:]

rows, cols, u, v = hungarian_min_cost(C, verbose=True)
print('\nassignment:', list(zip(rows, cols)))
print('total cost:', C[rows, cols].sum())
print('u:', u)
print('v:', v)
print('final slack:\n', C - u[:, None] - v[None, :])

## 6. KKT 验证与 SciPy 交叉验证

最后检查三件事：

- final slack 是否非负（对偶可行）；
- 所有选中边的 slack 是否为零（互补松弛）；
- 原始目标是否等于对偶目标（强对偶）。

`scipy.optimize.linear_sum_assignment` 也是工程中常用的精确线性分配求解器；它只用于交叉验证，前面的实现本身不依赖 SciPy。

In [ ]:
slack = C - u[:, None] - v[None, :]
primal = C[rows, cols].sum()
dual = u.sum() + v.sum()

assert np.all(slack >= -1e-9), '对偶不可行'
assert np.allclose(slack[rows, cols], 0.0), '选中边不满足互补松弛'
assert np.isclose(primal, dual), '原始与对偶目标不相等'
print(f'primal objective = {primal:.1f}')
print(f'dual objective   = {dual:.1f}')
print('KKT checks passed.')

try:
    from scipy.optimize import linear_sum_assignment
    scipy_rows, scipy_cols = linear_sum_assignment(C)
    print('SciPy assignment:', list(zip(scipy_rows, scipy_cols)))
    print('SciPy cost:', C[scipy_rows, scipy_cols].sum())
except ImportError:
    print('SciPy 未安装；纯 NumPy 实现仍已完成验证。')

## 7. MOT 中的矩形矩阵与 gating

实际 MOT 往往有 $N_{track}
eq N_{det}$。常见做法是加入 dummy 行/列，dummy 匹配意味着 unmatched track 或 new detection；另外先用 IoU、Mahalanobis distance 或 ReID 阈值进行 gating，将不可能的边赋予很大的有限代价。匹配结束后，应丢弃所有 dummy 匹配和代价超过 gate 的匹配。

Hungarian 的时间复杂度为 $O(n^3)$（方阵边长为 $n$）。通常 MOT 每帧关联对象数量有限，且 gating 会显著缩小候选集合。